In [17]:
# tells Python where the project/data/database are

from pathlib import Path
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_CURATED = PROJECT_ROOT / "data" / "curated"
DB_PATH = PROJECT_ROOT / "data" / "energy.duckdb"

print("Project root:", PROJECT_ROOT)
print("Curated folder:", DATA_CURATED)
print("DuckDB:", DB_PATH)



Project root: c:\Users\Uporabnik\Energy-Analytics
Curated folder: c:\Users\Uporabnik\Energy-Analytics\data\curated
DuckDB: c:\Users\Uporabnik\Energy-Analytics\data\energy.duckdb


In [18]:
# lists CSV files

for file in DATA_CURATED.glob("*.csv"):
    print(file.name)

arso_temp_ljubljana_bezigrad_2025_clean.csv
balancing_volumes_si_clean.csv
da_prices_si_2025_clean.csv
da_prices_si_2025_daily_avg.csv
da_prices_si_clean.csv
da_price_vs_arso_temp_2025.csv
imbalance_si_clean.csv


In [19]:
#Load all base clean datasets

da_2025 = pd.read_csv(
    DATA_CURATED / "da_prices_si_2025_clean.csv",
    parse_dates=["timestamp"]
)

da_2026 = pd.read_csv(
    DATA_CURATED / "da_prices_si_clean.csv",
    parse_dates=["timestamp"]
)

imbalance = pd.read_csv(
    DATA_CURATED / "imbalance_si_clean.csv",
    parse_dates=["timestamp"]
)

balancing = pd.read_csv(
    DATA_CURATED / "balancing_volumes_si_clean.csv",
    parse_dates=["timestamp"]
)

weather = pd.read_csv(
    DATA_CURATED / "arso_temp_ljubljana_bezigrad_2025_clean.csv",
    parse_dates=["date"]
)

In [20]:
# Market Time Unit (MTU): 60min before 2025-10-01, 15min from 2025-10-01 onward.

da_2025["source_file"] = "da_prices_si_2025_clean.csv"
da_2026["source_file"] = "da_prices_si_clean.csv"

da_2025["mtu"] = da_2025["timestamp"].apply(
    lambda x: "60min" if x < pd.Timestamp("2025-10-01") else "15min"
)

da_2026["mtu"] = "15min"

In [21]:
#Combine DA prices

da_all = pd.concat([da_2025, da_2026], ignore_index=True)

da_all = (
    da_all
    .drop_duplicates(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Rows:", len(da_all))
print("Start:", da_all["timestamp"].min())
print("End:", da_all["timestamp"].max())
print("Duplicates:", da_all["timestamp"].duplicated().sum())

da_all.head()

Rows: 23923
Start: 2025-01-01 00:00:00
End: 2026-04-01 00:00:00
Duplicates: 0


,timestamp,price_eur_mwh,source_file,mtu
0,2025-01-01 00:00:00,118.46,da_prices_si_2025_clean.csv,60min
1,2025-01-01 01:00:00,129.07,da_prices_si_2025_clean.csv,60min
2,2025-01-01 02:00:00,121.10,da_prices_si_2025_clean.csv,60min
3,2025-01-01 03:00:00,94.28,da_prices_si_2025_clean.csv,60min
4,2025-01-01 04:00:00,63.69,da_prices_si_2025_clean.csv,60min


In [23]:
da_all["mtu"].value_counts()

mtu
15min    17372
60min     6551
Name: count, dtype: int64

In [24]:
da_all[
    (da_all["timestamp"] >= "2025-09-30 20:00") &
    (da_all["timestamp"] <= "2025-10-01 02:00")
]

,timestamp,price_eur_mwh,source_file,mtu
6547,2025-09-30 20:00:00,159.94,da_prices_si_2025_clean.csv,60min
6548,2025-09-30 21:00:00,127.56,da_prices_si_2025_clean.csv,60min
6549,2025-09-30 22:00:00,106.97,da_prices_si_2025_clean.csv,60min
6550,2025-09-30 23:00:00,103.88,da_prices_si_2025_clean.csv,60min
6551,2025-10-01 00:00:00,109.10,da_prices_si_2025_clean.csv,15min
6552,2025-10-01 00:15:00,97.24,da_prices_si_2025_clean.csv,15min
6553,2025-10-01 00:30:00,96.22,da_prices_si_2025_clean.csv,15min
6554,2025-10-01 00:45:00,96.14,da_prices_si_2025_clean.csv,15min
6555,2025-10-01 01:00:00,92.51,da_prices_si_2025_clean.csv,15min
6556,2025-10-01 01:15:00,92.23,da_prices_si_2025_clean.csv,15min


In [26]:
da_all.to_csv(DATA_CURATED / "da_prices_si_all.csv", index=False)

print("Saved:", DATA_CURATED / "da_prices_si_all.csv")

Saved: c:\Users\Uporabnik\Energy-Analytics\data\curated\da_prices_si_all.csv


In [27]:
# Connect to the DuckDB database file.
# If the file does not exist yet, DuckDB will create it.
con = duckdb.connect(str(DB_PATH))

# Register pandas DataFrames as temporary DuckDB views.
# This lets DuckDB read the DataFrames directly.
con.register("da_all_df", da_all)
con.register("imbalance_df", imbalance)
con.register("balancing_df", balancing)
con.register("weather_df", weather)

# Create or replace permanent DuckDB tables from the registered DataFrames.
# These tables will be stored inside data/energy.duckdb.
con.execute("CREATE OR REPLACE TABLE da_prices_all AS SELECT * FROM da_all_df")
con.execute("CREATE OR REPLACE TABLE imbalance_prices AS SELECT * FROM imbalance_df")
con.execute("CREATE OR REPLACE TABLE balancing_volumes AS SELECT * FROM balancing_df")
con.execute("CREATE OR REPLACE TABLE weather_daily AS SELECT * FROM weather_df")

# Confirm that the write step finished.
print("Tables written to DuckDB.")

Tables written to DuckDB.


In [28]:
con.execute("SHOW TABLES").fetchdf()

,name
0,balancing_df
1,balancing_volumes
2,calendar
3,da_all_df
4,da_prices
5,da_prices_all
6,imbalance_df
7,imbalance_prices
8,weather_daily
9,weather_df


In [ ]:
# Check the main tables that use a timestamp column.

for table in ["da_prices_all", "imbalance_prices", "balancing_volumes"]:
    print("\n" + table)
    display(con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            MIN(timestamp) AS start_time,
            MAX(timestamp) AS end_time
        FROM {table}
    """).fetchdf())

# Check the weather table separately.
# Weather uses a date column, not a timestamp column.

print("\nweather_daily")
display(con.execute("""
    SELECT
        COUNT(*) AS rows,
        MIN(date) AS start_date,
        MAX(date) AS end_date
    FROM weather_daily
""").fetchdf())


da_prices_all


,rows,start_time,end_time
0,23923,2025-01-01,2026-04-01



imbalance_prices


,rows,start_time,end_time
0,11520,2025-11-01 00:15:00,2026-03-01



balancing_volumes


,rows,start_time,end_time
0,20160,2025-11-01 00:15:00,2026-03-01



weather_daily


,rows,start_date,end_date
0,365,2025-01-01,2025-12-31
